In [6]:
import time
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

In [7]:
def crawl_tripadvisor_interactive(): 
    options = Options()
    # Nhập hồn vào cổng 9222
    options.add_experimental_option("debuggerAddress", "127.0.0.1:9222")
    
    print("Đang 'nhập hồn' vào Chrome thật của bạn...")
    driver = webdriver.Chrome(options=options)
    print("Kết nối thành công! Bắt đầu cào dữ liệu...")
    
    all_reviews_data = []

    for step in range(1, 6):
        print("\n" + "="*50)
        print(f"--- BƯỚC {step}/5: CHỌN FILTER ---")
        
        rating_input = input("-> BƯỚC A: Bạn đang chọn bộ lọc MẤY SAO? (Nhập 1, 2, 3, 4, 5 - Nhập 0 để bỏ qua): ")
        try:
            current_rating = int(rating_input.strip())
            if current_rating == 0: continue
        except ValueError: continue

        user_input = input(f"-> BƯỚC B: Bạn muốn lấy BAO NHIÊU comments cho {current_rating} sao?: ")
        try:
            target_count = int(user_input.strip())
            if target_count <= 0: continue
        except ValueError: continue
            
        print(f"Bắt đầu thu thập {target_count} đánh giá {current_rating} sao...")
        current_count = 0
        page_count = 0
        
        while current_count < target_count:
            page_count += 1
            print(f"\n⏳ Đang quét dữ liệu trang {page_count}...")
            time.sleep(random.uniform(3.5, 5.5)) 
            
            # Quét các thẻ review
            review_cards = driver.find_elements(By.CSS_SELECTOR, "div[data-automation='reviewCard']")
            if not review_cards:
                print("Không tìm thấy review nào trên trang này.")
                break
                
            for card in review_cards:
                if current_count >= target_count: break 
                    
                try:
                    # 1. Lấy Username
                    try:
                        profile_links = card.find_elements(By.XPATH, ".//a[contains(@href, '/Profile/')]")
                        username = next((link.text.strip() for link in profile_links if link.text.strip()), "Khách ẩn danh")
                    except: username = "Khách ẩn danh"    
                        
                    # 2. Lấy Rating
                    rating = str(current_rating)
                    
                    # 3. BẤM "READ MORE" CHO TỪNG BÀI VÀ LẤY COMMENT
                    comment = ""
                    try:
                        # Chỉ tìm và bấm nút Xem thêm của RIÊNG BÀI HIỆN TẠI
                        read_more_btn = card.find_elements(By.XPATH, ".//span[contains(text(), 'Read more') or contains(text(), 'Xem thêm') or contains(@class, 'yMvVv')]")
                        if read_more_btn:
                            driver.execute_script("arguments[0].click();", read_more_btn[0])
                            time.sleep(random.uniform(0.5, 1.0)) # Chờ xíu cho popup bung ra
                            
                        # Lấy chữ từ Popup (nếu nó đang mở)
                        popup_text = driver.find_elements(By.XPATH, "//div[@role='dialog']//q | //div[@role='dialog']//span[string-length(text()) > 15]")
                        if popup_text and len(popup_text[0].text.strip()) > 10:
                            comment = popup_text[0].text.strip()
                        else:
                            # Nếu không có popup thì lấy chữ trực tiếp từ trên trang
                            elements = card.find_elements(By.XPATH, ".//span | .//q")
                            valid_texts = []
                            blacklist = ["written", "subjective opinion", "tripadvisor performs checks", "management representative", "management board", "vinwonders", "dear", "warm greetings", "thank you", "xin chao", "salutations"]
                            
                            for el in elements:
                                text = el.text.strip()
                                is_spam = any(bad_word in text.lower() for bad_word in blacklist)
                                if len(text) > 15 and not is_spam: valid_texts.append(text)
                                    
                            if valid_texts: comment = max(valid_texts, key=len)
                                
                        comment = comment.replace("\n", " ") if comment else "Không có nội dung"
                    except Exception as e:
                        comment = "Không có nội dung"

                    # DỌN DẸP NGAY: Bấm ESC để đóng sạch sẽ cái Popup vừa mở trước khi qua bài mới
                    webdriver.ActionChains(driver).send_keys('\ue00c').perform() 
                    time.sleep(random.uniform(0.2, 0.4))
                        
                    # 4. Gán tên địa điểm
                    address = "Po Nagar Cham Towers"
                    
                    all_reviews_data.append({
                        "Username": username,
                        "Address": address, 
                        "Rating": rating,
                        "Comment": comment
                    })
                    current_count += 1
                    
                except Exception as e:
                    continue 
            
            print(f"  => Tiến độ: Đã gom được {current_count}/{target_count} đánh giá.")
            
            # --- CHUYỂN TRANG ---
            if current_count < target_count:
                try:
                    next_button = driver.find_element(By.XPATH, "//a[contains(@aria-label, 'Next')] | //a[contains(@aria-label, 'Tiếp')] | //*[@data-smoke-attr='pagination-next-arrow'] | //a[contains(@class, 'next')] | //div[contains(@class, 'pagination')]//a[last()]")
                    if not next_button.get_attribute("href") or "disabled" in next_button.get_attribute("class"):
                        print("Đã đến trang cuối. Không thể lấy thêm.")
                        break
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center', inline: 'nearest'});", next_button)
                    time.sleep(random.uniform(1.2, 2.5)) 
                    driver.execute_script("arguments[0].click();", next_button)
                    time.sleep(random.uniform(4.5, 7.5)) 
                except:
                    print("Không tìm thấy nút mũi tên chuyển trang.")
                    break 

    print("Hoàn thành!")
    return pd.DataFrame(all_reviews_data)

In [8]:
# === ĐOẠN CHẠY CODE VÀ LƯU FILE ===
if __name__ == "__main__":
    df_result = crawl_tripadvisor_interactive() 
    
    # Rút thẳng tên địa điểm từ cột "Address" của dòng đầu tiên trong dữ liệu vừa cào
    if not df_result.empty:
        address = df_result["Address"].iloc[0]
    else:
        address = "Du_lieu_trong" # Đề phòng trường hợp cào bị lỗi không có data
        
    # Tự động thay khoảng trắng bằng dấu gạch dưới cho chuẩn tên file
    safe_filename = address.replace(" ", "_")
    
    try:
        df_result.to_excel(f"{safe_filename}_reviews.xlsx", index=False)
        print(f"Đã lưu dữ liệu thành công ra file Excel: {safe_filename}_reviews.xlsx")
    except ModuleNotFoundError:
        df_result.to_csv(f"{safe_filename}_reviews.csv", index=False, encoding='utf-8-sig')
        print(f"Đã lưu dữ liệu tại: {safe_filename}_reviews.csv")

Đang 'nhập hồn' vào Chrome thật của bạn...
Kết nối thành công! Bắt đầu cào dữ liệu...

--- BƯỚC 1/5: CHỌN FILTER ---
Bắt đầu thu thập 21 đánh giá 1 sao...

⏳ Đang quét dữ liệu trang 1...
  => Tiến độ: Đã gom được 10/21 đánh giá.

⏳ Đang quét dữ liệu trang 2...
  => Tiến độ: Đã gom được 20/21 đánh giá.

⏳ Đang quét dữ liệu trang 3...
  => Tiến độ: Đã gom được 21/21 đánh giá.

--- BƯỚC 2/5: CHỌN FILTER ---
Bắt đầu thu thập 75 đánh giá 2 sao...

⏳ Đang quét dữ liệu trang 1...
  => Tiến độ: Đã gom được 10/75 đánh giá.

⏳ Đang quét dữ liệu trang 2...
  => Tiến độ: Đã gom được 20/75 đánh giá.

⏳ Đang quét dữ liệu trang 3...
  => Tiến độ: Đã gom được 30/75 đánh giá.

⏳ Đang quét dữ liệu trang 4...
  => Tiến độ: Đã gom được 40/75 đánh giá.

⏳ Đang quét dữ liệu trang 5...
  => Tiến độ: Đã gom được 50/75 đánh giá.

⏳ Đang quét dữ liệu trang 6...
  => Tiến độ: Đã gom được 60/75 đánh giá.

⏳ Đang quét dữ liệu trang 7...
  => Tiến độ: Đã gom được 70/75 đánh giá.

⏳ Đang quét dữ liệu trang 8...
  =>